## Training and Validation Set data preparation

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# pd.set_option("display.max_colwidth", None)

In [2]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
df = pd.read_json("finfact.json")

In [4]:
df.head()

,url,claim,author,posted,sci_digest,justification,issues,image_data,evidence,label,visualization_bias
0,https://www.politifact.com/factchecks/2023/jul...,Video shows that George Soros is going bankrupt.,Ciara O'Rourke,07/20/2023,[We found no evidence that billionaire philant...,[A recent Facebook post claims that billionair...,"[Bankruptcy, Facebook Fact-checks]",[],"[{'sentence': 'IT HAPPENED, the July 18postsay...",false,NaN
1,https://www.politifact.com/factchecks/2016/dec...,"Because of Obamacare, Medicare is going broke.",Tom Kertscher,12/23/2016,[],[As U.S. House SpeakerPaul Ryandiscussed the r...,"[Bankruptcy, Federal Budget, Health Care, Medi...",[],[{'sentence': 'As U.S. House SpeakerPaul Ryand...,false,NaN
2,https://www.politifact.com/factchecks/2016/oct...,"Back in the Great Recession, when millions of ...",Lauren Carroll,10/18/2016,[],[Donald Trump didnt care about rescuing the au...,"[National, Bankruptcy, Candidate Biography, Jobs]",[],[{'sentence': 'Trumps position on an auto bail...,false,NaN
3,https://www.politifact.com/factchecks/2016/jul...,Says Donald Trump has bankrupt four separate b...,Sean Gorman,07/11/2016,[],[U.S. Sen. Mark Warner says presumptive Democr...,"[Bankruptcy, Candidate Biography, Gambling, Vi...",[],[{'sentence': 'To think that Mr. Trump is tryi...,true,NaN
4,https://www.politifact.com/factchecks/2016/may...,"In 2006, Donald Trump was hoping for a real es...",C. Eugene Emery Jr.,05/26/2016,[],[The commercial seems like an example of the o...,"[National, Bankruptcy, Candidate Biography, De...",[],[{'sentence': 'Democrat Hillary Clinton posted...,true,NaN


In [5]:
new_df = df[["claim", "justification", "evidence", "label"]]

In [6]:
def process_evidence(x):
    evidences = []
    for evidence in x:
        evidences.append(evidence["sentence"])
    return " ".join(evidences)

def prepare_prompt(sample, tokenizer):
    claim, justification, evidence, label = sample["claim"], sample["justification"], sample["evidence"], sample["label"]
    justification = " ".join(justification)
    evidence = process_evidence(evidence)
    prompt = f"You are a financial analyst. Your task is to analyse if the justification and evidence are aligned or not. You are provided with a claim: {claim} Justification for the claim: {justification} Evidence for validation of claim: {evidence}"
    text = ""
    text += f"{tokenizer.bos_token}[INST] {prompt} [/INST]"
    text += f" {label}{tokenizer.eos_token}"
    return text

In [7]:
new_df["text"] = new_df.apply(prepare_prompt, args=(tokenizer,), axis=1)

/tmp/ipykernel_15699/743248520.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["text"] = new_df.apply(prepare_prompt, args=(tokenizer,), axis=1)


In [8]:
train_data, test_ds = train_test_split(new_df, shuffle=True, test_size=0.1, random_state=42)
train_ds, val_ds = train_test_split(train_data, shuffle=True, test_size=0.1, random_state=42)

print(f"Train dataset {len(train_ds)}, and Validation dataset {len(val_ds)}, and Test dataset: {len(test_ds)}")

Train dataset 2884, and Validation dataset 321, and Test dataset: 357


In [9]:
val_ds.head()

,claim,justification,evidence,label,text
2221,We already have $23 billion worth of debt.,"[Florida Gov., Rick Scott vetoed a record $615...",[{'sentence': 'Florida Gov. Rick Scott vetoed ...,true,<s>[INST] You are a financial analyst. Your ta...
1028,If the Dow Joans ever falls more than 1000 poi...,"[As the Dow JonesplungedFeb., 5, not long afte...","[{'sentence': 'As the Dow JonesplungedFeb. 5, ...",false,<s>[INST] You are a financial analyst. Your ta...
2719,Bill Nelson actually voted in favor of higher ...,"[U.S. Rep. Connie Mack IV, R-Fort Myers, has r...","[{'sentence': 'U.S. Rep. Connie Mack IV, R-For...",false,<s>[INST] You are a financial analyst. Your ta...
441,Says Donald Trump's first 17 Cabinet appointme...,[Critics of President-electDonald Trumpsay he ...,[{'sentence': 'Critics of President-electDonal...,true,<s>[INST] You are a financial analyst. Your ta...
2056,"In 2012, 1 in 4 Wisconsin schools had a subpar...",[Assembly Speaker Robin Vos (R-Rochester) igni...,"[{'sentence': 'In 2012, 1 in 4 Wisconsin schoo...",false,<s>[INST] You are a financial analyst. Your ta...


In [10]:
train_ds.head()

,claim,justification,evidence,label,text
1384,"In the past year alone, Ohio businesses have c...",[The nasty political ads have been gone for we...,[{'sentence': 'The nasty political ads have be...,true,<s>[INST] You are a financial analyst. Your ta...
2543,We can fix our roads without raising taxes.,[Illinois voters are being asked whether they ...,[{'sentence': 'A coalition called Citizens to ...,false,<s>[INST] You are a financial analyst. Your ta...
3162,"Unemployment now pays $24/hour, even if your w...",[A $600-per-week boost in unemployment benefit...,[{'sentence': 'As the bill neared final approv...,neutral,<s>[INST] You are a financial analyst. Your ta...
1641,"Understand, this is unemployment insurance. It...",[If you lose your job and its not your fault -...,[{'sentence': 'If you lose your job and its no...,true,<s>[INST] You are a financial analyst. Your ta...
275,Tulsi Gabbard Venmos Nancy Pelosi $600.01 forc...,[Some social media users are earnestly sharing...,[{'sentence': 'BREAKING: Tulsi Gabbard Venmos ...,false,<s>[INST] You are a financial analyst. Your ta...


In [11]:
train_ds.to_csv("train_df.csv", index=False)
val_ds.to_csv("val_df.csv", index=False)

## Test set data preparation

In [12]:
test_ds.head()

,claim,justification,evidence,label,text
2634,When the New Hampshire Legislature raised the ...,[New Hampshire voters are famously unenthusias...,[{'sentence': 'Thead-- part of what the group ...,false,<s>[INST] You are a financial analyst. Your ta...
184,"The fundraising numbers are in, and our grassr...",[When Republican moderateTom Petriretires next...,[{'sentence': 'When Republican moderateTom Pet...,false,<s>[INST] You are a financial analyst. Your ta...
1847,West Virginia spends more tax dollars on publi...,[Does West Virginia rank among the top quarter...,"[{'sentence': 'In the April 5thread, the party...",false,<s>[INST] You are a financial analyst. Your ta...
1361,Says Travis County's unemployment rate is belo...,"[CORRECTION, 2:58 p.m., Feb. 25, 2013:This art...",[{'sentence': 'A Travis County commissioner wh...,true,<s>[INST] You are a financial analyst. Your ta...
2855,"Fifty-one percent -- that is, a majority of Am...","[In aJuly 7, 2011, floor speech, Sen. John Cor...","[{'sentence': 'In aJuly 7, 2011, floor speech,...",true,<s>[INST] You are a financial analyst. Your ta...


In [13]:
test_ds.dropna(inplace=True)

In [14]:
test_ds.drop(columns=["text"], inplace=True)

In [15]:
test_ds.head()

,claim,justification,evidence,label
2634,When the New Hampshire Legislature raised the ...,[New Hampshire voters are famously unenthusias...,[{'sentence': 'Thead-- part of what the group ...,false
184,"The fundraising numbers are in, and our grassr...",[When Republican moderateTom Petriretires next...,[{'sentence': 'When Republican moderateTom Pet...,false
1847,West Virginia spends more tax dollars on publi...,[Does West Virginia rank among the top quarter...,"[{'sentence': 'In the April 5thread, the party...",false
1361,Says Travis County's unemployment rate is belo...,"[CORRECTION, 2:58 p.m., Feb. 25, 2013:This art...",[{'sentence': 'A Travis County commissioner wh...,true
2855,"Fifty-one percent -- that is, a majority of Am...","[In aJuly 7, 2011, floor speech, Sen. John Cor...","[{'sentence': 'In aJuly 7, 2011, floor speech,...",true


In [16]:
def prepare_test_prompt(sample, tokenizer):
    claim, justification, evidence = sample["claim"], sample["justification"], sample["evidence"]
    justification = " ".join(justification)
    evidence = process_evidence(evidence)
    prompt = f"You are a financial analyst. Your task is to analyse if the justification and evidence are aligned or not. You are provided with a claim: {claim} Justification for the claim: {justification} Evidence for validation of claim: {evidence}"
    text = ""
    text += f"{tokenizer.bos_token}[INST] {prompt} [/INST]"
    return text

In [17]:
test_ds["text"] = test_ds.apply(prepare_test_prompt, args=(tokenizer,), axis=1)

In [18]:
test_ds.head()

,claim,justification,evidence,label,text
2634,When the New Hampshire Legislature raised the ...,[New Hampshire voters are famously unenthusias...,[{'sentence': 'Thead-- part of what the group ...,false,<s>[INST] You are a financial analyst. Your ta...
184,"The fundraising numbers are in, and our grassr...",[When Republican moderateTom Petriretires next...,[{'sentence': 'When Republican moderateTom Pet...,false,<s>[INST] You are a financial analyst. Your ta...
1847,West Virginia spends more tax dollars on publi...,[Does West Virginia rank among the top quarter...,"[{'sentence': 'In the April 5thread, the party...",false,<s>[INST] You are a financial analyst. Your ta...
1361,Says Travis County's unemployment rate is belo...,"[CORRECTION, 2:58 p.m., Feb. 25, 2013:This art...",[{'sentence': 'A Travis County commissioner wh...,true,<s>[INST] You are a financial analyst. Your ta...
2855,"Fifty-one percent -- that is, a majority of Am...","[In aJuly 7, 2011, floor speech, Sen. John Cor...","[{'sentence': 'In aJuly 7, 2011, floor speech,...",true,<s>[INST] You are a financial analyst. Your ta...


In [19]:
test_ds.to_csv("test_df.csv", index=False)